<a href="https://github.com/N3iKos/segsmaker-prallel">
  <img alt="GitHub repo" src="https://img.shields.io/badge/GitHub-6e5494?style=for-the-badge&logo=github&logoColor=white"/>
</a><br>

*   get your civitai api key from [here](https://civitai.com/user/account)


In [ ]:
# @title <b><font color='orange'>WebUI Installer</font></b> {"display-mode":"form"}

Webui = 'A1111' # @param ["A1111", "Forge", "ReForge", "ReForge-old", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI"]
Civitai___Key = '' # @param { type: "string", placeholder: "Your Civitai API Key (required)" }
HF_Read_Token = '' # @param { type: "string", placeholder: "Your Huggingface READ Token (optional)" }
Mount__GDrive = 'No' # @param ["Yes", "No"]
Enable_Parallel_Setup = True # @param { type: "boolean" }
Setup_Max_Parallel_Downloads = 3 # @param { type: "integer" }

Setup_Max_Parallel_Downloads = max(1, min(int(Setup_Max_Parallel_Downloads), 8))
mount = Mount__GDrive

if mount == 'Yes':
    from google.colab import drive
    drive.mount('/content/drive')

!curl -sLo /content/setup.py https://github.com/N3iKos/segsmaker-prallel/raw/main/script/KC/setup.py
%run /content/setup.py --webui="$Webui" --civitai_key="$Civitai___Key" --hf_read_token="$HF_Read_Token" --parallel_downloads="$Enable_Parallel_Setup" --max_parallel_downloads="$Setup_Max_Parallel_Downloads"

if mount == 'Yes':
    from pathlib import Path

    d = Path('/content/drive/MyDrive/Segsmaker')

    for n, p in {'checkpoint': CKPT, 'lora': LORA, 'vae': VAE, 'embeddings': Embeddings}.items():
        f = d / n
        f.mkdir(parents=True, exist_ok=True)
        s = p / f'drive-{n}'
        s.symlink_to(f, target_is_directory=True)

    !rm -rf $WebUI_Output
    o = d / {'ComfyUI': 'comfyui-output', 'SwarmUI': 'swarmui-output'}.get(Webui, 'output')
    o.mkdir(parents=True, exist_ok=True)
    WebUI_Output.symlink_to(o, target_is_directory=True)

    if Webui not in {'ComfyUI', 'SwarmUI'}:
        wc = WebUI / 'cache'
        !rm -rf $wc
        c = d / 'cache'
        c.mkdir(parents=True, exist_ok=True)
        wc.symlink_to(c, target_is_directory=True)


In [ ]:
# @title <b><font color='orange'>Optional Assets</font></b> {"display-mode":"form"}

Enable_Parallel_Optional_Downloads = True # @param { type: "boolean" }
Optional_Max_Parallel_Downloads = 3 # @param { type: "integer" }
Extension_or_Node_Repo_1 = '' # @param { type: "string", placeholder: "Optional git repo URL" }
Extension_or_Node_Repo_2 = '' # @param { type: "string", placeholder: "Optional git repo URL" }
VAE_URL = '' # @param { type: "string", placeholder: "Optional VAE URL" }
Embedding_URL = '' # @param { type: "string", placeholder: "Optional embedding URL" }
Upscaler_URL = '' # @param { type: "string", placeholder: "Optional upscaler URL" }
Flux_UNET_URL = '' # @param { type: "string", placeholder: "Optional FLUX UNet URL" }
Flux_CLIP_URL = '' # @param { type: "string", placeholder: "Optional FLUX CLIP URL" }

from nenen88 import clone, download_many

Optional_Max_Parallel_Downloads = max(1, min(int(Optional_Max_Parallel_Downloads), 8))

repos = [repo.strip() for repo in [Extension_or_Node_Repo_1, Extension_or_Node_Repo_2] if repo.strip()]
if repos:
    %cd -q $Extensions
    for repo in repos:
        clone(repo)

optional_downloads = []
for url, target in [
    (VAE_URL, VAE),
    (Embedding_URL, Embeddings),
    (Upscaler_URL, Upscalers),
    (Flux_UNET_URL, UNET),
    (Flux_CLIP_URL, CLIP),
]:
    value = str(url).strip()
    if value:
        optional_downloads.append(f'{value} {target}')

if optional_downloads:
    download_many(
        optional_downloads,
        max_workers=Optional_Max_Parallel_Downloads,
        parallel=Enable_Parallel_Optional_Downloads,
    )
else:
    print('[INFO] No optional asset URL provided.')


In [ ]:
# @title <b><font color='orange'>Download Model / LoRA</font></b> {"display-mode":"form"}

Enable_Parallel_Model_LoRA_Downloads = True # @param { type: "boolean" }
Model_LoRA_Max_Parallel_Downloads = 3 # @param { type: "integer" }
Model_1 = 'https://huggingface.co/pantat88/back_up/resolve/main/bigblu25dmix25DStyle_v10.safetensors' # @param { type: "string", placeholder: "Model URL or URL filename.safetensors" }
Model_2 = '' # @param { type: "string", placeholder: "Model URL or URL filename.safetensors" }
Model_3 = '' # @param { type: "string", placeholder: "Model URL or URL filename.safetensors" }
Model_4 = '' # @param { type: "string", placeholder: "Model URL or URL filename.safetensors" }
Model_5 = '' # @param { type: "string", placeholder: "Model URL or URL filename.safetensors" }
LoRA_1 = 'https://civitai.com/models/122359/detail-tweaker-xl' # @param { type: "string", placeholder: "LoRA URL or URL filename.safetensors" }
LoRA_2 = 'https://civitai.com/models/669571/pony-add-more-details details-add-more-pony.safetensors' # @param { type: "string", placeholder: "LoRA URL or URL filename.safetensors" }
LoRA_3 = '' # @param { type: "string", placeholder: "LoRA URL or URL filename.safetensors" }
LoRA_4 = '' # @param { type: "string", placeholder: "LoRA URL or URL filename.safetensors" }
LoRA_5 = '' # @param { type: "string", placeholder: "LoRA URL or URL filename.safetensors" }

from nenen88 import download_many

Model_LoRA_Max_Parallel_Downloads = max(1, min(int(Model_LoRA_Max_Parallel_Downloads), 8))

def with_target(value, target):
    value = str(value).strip()
    if not value:
        return ''
    parts = value.split()
    if len(parts) == 1:
        return f'{value} {target}'
    return f"{parts[0]} {target} {' '.join(parts[1:])}"

model_inputs = [Model_1, Model_2, Model_3, Model_4, Model_5]
lora_inputs = [LoRA_1, LoRA_2, LoRA_3, LoRA_4, LoRA_5]
requests = [with_target(item, CKPT) for item in model_inputs]
requests += [with_target(item, LORA) for item in lora_inputs]

requests = [item for item in requests if item]
if requests:
    download_many(
        requests,
        max_workers=Model_LoRA_Max_Parallel_Downloads,
        parallel=Enable_Parallel_Model_LoRA_Downloads,
    )
else:
    print('[INFO] No Model or LoRA URL provided.')


In [ ]:
''' Controlnet '''
%run $Controlnet_Widget

## Launcher

Use the form cell below. The preset dropdown automatically maps the selected or installed WebUI to recommended launch arguments.


In [ ]:
# @title <b><font color='orange'>Run WebUI</font></b> {"display-mode":"form"}

Launch_Preset = 'Recommended for installed WebUI' # @param ["Recommended for installed WebUI", "A1111", "Forge", "ReForge", "ReForge-old", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI", "Custom Only"]
Custom_Launch_Args = '' # @param { type: "string", placeholder: "Optional extra or replacement args" }
Tunnel = 'Auto' # @param ["Auto", "NGROK", "ZROK"]
NGROK_Token = '' # @param { type: "string", placeholder: "Optional NGROK token" }
ZROK_Token = '' # @param { type: "string", placeholder: "Optional ZROK token" }
Skip_ComfyUI_Check = False # @param { type: "boolean" }

import json
import shlex
from pathlib import Path

try:
    from KANDANG import HOMEPATH
except Exception:
    HOMEPATH = '/content'

marking = Path(HOMEPATH) / 'gutris1/marking.json'
installed_ui = Webui if 'Webui' in globals() else None
if marking.exists():
    installed_ui = json.loads(marking.read_text()).get('ui', installed_ui)

recommended_args = {
    'A1111': '--xformers',
    'Forge': '--disable-xformers --opt-sdp-attention --cuda-stream',
    'ReForge': '--xformers --cuda-stream',
    'ReForge-old': '--xformers --cuda-stream',
    'Forge-Classic': '--xformers --cuda-stream --persistent-patches',
    'Forge-Neo': '--xformers --cuda-malloc --cuda-stream',
    'ComfyUI': '--dont-print-server --use-pytorch-cross-attention',
    'SwarmUI': '--launch_mode none',
}

if Launch_Preset == 'Recommended for installed WebUI':
    final_args = recommended_args.get(installed_ui, '')
elif Launch_Preset == 'Custom Only':
    final_args = ''
else:
    final_args = recommended_args.get(Launch_Preset, '')

if Custom_Launch_Args.strip():
    final_args = f'{final_args} {Custom_Launch_Args.strip()}'.strip()

if Skip_ComfyUI_Check:
    final_args = f'{final_args} --skip-comfyui-check'.strip()

if Tunnel == 'NGROK' and NGROK_Token.strip():
    final_args = f'{final_args} --N={shlex.quote(NGROK_Token.strip())}'.strip()
elif Tunnel == 'ZROK' and ZROK_Token.strip():
    final_args = f'{final_args} --Z={shlex.quote(ZROK_Token.strip())}'.strip()

%cd -q $WebUI
# Equivalent form command: %run segsmaker.py
get_ipython().run_line_magic('run', f'segsmaker.py {final_args}')
